# TürkResearcher — DergiPark Embedding (Append to Chroma)

This notebook **adds 106K Turkish journal articles** (harvested from DergiPark via OAI-PMH) to the existing 633K-thesis Chroma index, producing a unified 740K-record corpus.

**Strategy:** download the existing Chroma DB from HF Hub, embed the new DergiPark parquet with the SAME mpnet-base-v2 model + cosine distance, `upsert` into the SAME `turkish_theses` collection, push back to HF Hub.

**Pre-requisites (already done locally):**
- `dergipark_filtered.parquet` (173 MB, 106,641 records) uploaded to `hakansabunis/tr-academic-research-agent-index`.

**Akış:**
1. Setup + GPU + HF auth
2. Drive mount
3. Pull existing Chroma + DergiPark parquet from HF Hub
4. Append new records (with `source_type=article`)
5. Smoke retrieval test (mixed thesis + article)
6. Push updated Chroma back to Hub

## 1. Setup

In [ ]:
!nvidia-smi
!pip install -q chromadb sentence-transformers datasets huggingface_hub pyarrow tqdm

## 2. Workspace setup (use Colab's /content disk — fast, 100 GB)

We deliberately avoid Drive: the 14.8 GB Chroma + 173 MB parquet + headroom for embedding would exceed Drive's free 15 GB tier. `/content` (Colab local disk) is faster and has 100 GB. Disconnect = lose state, but the notebook is **resumable**: re-run, the `upsert` is idempotent.

In [ ]:
import os

WORK_DIR = '/content/turkresearcher_dp'
os.makedirs(WORK_DIR, exist_ok=True)
print(f'Working dir: {WORK_DIR}')

!df -h /content | tail -1

## 3. HF auth + config

In [ ]:
from getpass import getpass
from huggingface_hub import login

hf_token = getpass('HF write token: ')
login(hf_token)

In [ ]:
HF_REPO          = 'hakansabunis/tr-academic-research-agent-index'
EMBED_MODEL      = 'sentence-transformers/paraphrase-multilingual-mpnet-base-v2'
COLLECTION_NAME  = 'turkish_theses'  # SAME collection — append, not new
BATCH_SIZE       = 256

PERSIST_DIR        = f'{WORK_DIR}/chroma_db'
DERGIPARK_PARQUET  = f'{WORK_DIR}/dergipark_filtered.parquet'

## 4. Pull existing Chroma + DergiPark parquet from HF Hub

First run downloads ~14.8 GB Chroma + 173 MB parquet. CDN typically delivers in ~5-15 min.

In [ ]:
from huggingface_hub import snapshot_download
import shutil

if os.path.exists(PERSIST_DIR) and os.listdir(PERSIST_DIR):
    print(f'Already have local Chroma at {PERSIST_DIR} (size: '
          f'{sum(os.path.getsize(os.path.join(r,f)) for r,_,files in os.walk(PERSIST_DIR) for f in files)/1024**3:.1f} GB)')
else:
    print(f'Downloading {HF_REPO} → {WORK_DIR}/_hub ...')
    cache = f'{WORK_DIR}/_hub'
    os.makedirs(cache, exist_ok=True)
    snapshot_download(repo_id=HF_REPO, repo_type='dataset', local_dir=cache, token=hf_token)

    # Move chroma_db and parquet into WORK_DIR
    if os.path.exists(f'{cache}/chroma_db'):
        shutil.move(f'{cache}/chroma_db', PERSIST_DIR)
    if os.path.exists(f'{cache}/dergipark_filtered.parquet'):
        shutil.move(f'{cache}/dergipark_filtered.parquet', DERGIPARK_PARQUET)
    print('Pull done.')

print('Chroma:', os.path.exists(PERSIST_DIR))
print('Parquet:', os.path.exists(DERGIPARK_PARQUET))

## 5. Append DergiPark records to existing Chroma

Resumable: existing tez_no values (e.g. `art-1704902`) are upserted, so re-running this cell is idempotent.

In [ ]:
import chromadb
from chromadb.utils import embedding_functions
from tqdm.auto import tqdm
import pandas as pd
import time

client = chromadb.PersistentClient(path=PERSIST_DIR)

print(f'Loading embedder on GPU: {EMBED_MODEL}')
embed_fn = embedding_functions.SentenceTransformerEmbeddingFunction(
    model_name=EMBED_MODEL, device='cuda'
)
coll = client.get_or_create_collection(
    name=COLLECTION_NAME,
    embedding_function=embed_fn,
    metadata={'hnsw:space': 'cosine'},
)

print(f'Existing records in collection: {coll.count():,}')

df = pd.read_parquet(DERGIPARK_PARQUET)
print(f'DergiPark records to add: {len(df):,}')

def _doc(r):
    t = (r.get('title_tr') or '').strip()
    a = (r.get('abstract_tr') or '').strip()
    return f'{t}\n\n{a}' if t else a

def _meta(r):
    def s(k):
        v = r.get(k)
        if v is None or (isinstance(v, float) and pd.isna(v)):
            return ''
        return str(v)
    try:
        year_int = int(r.get('year') or 0)
    except (TypeError, ValueError):
        year_int = 0
    return {
        'tez_no':      s('tez_no'),
        'source_type': 'article',     # NEW field, distinguishes from theses
        'title_tr':    s('title_tr'),
        'title_en':    s('title_en'),
        'author':      s('author'),
        'advisor':     '',
        'location':    s('location'),  # publisher
        'subject':     s('subject'),
        'year':        year_int,
        'pages':       0,
        'degree':      '',
        'pdf_url':     s('pdf_url'),
        'language':    s('language'),
        'journal':     s('journal'),  # extra field for articles
    }

t0 = time.time()
for i in tqdm(range(0, len(df), BATCH_SIZE), desc='upserting'):
    batch = df.iloc[i:i+BATCH_SIZE]
    ids = batch['tez_no'].tolist()
    docs = [_doc(r) for _, r in batch.iterrows()]
    metas = [_meta(r) for _, r in batch.iterrows()]
    coll.upsert(ids=ids, documents=docs, metadatas=metas)

print(f'\nDone in {(time.time()-t0)/60:.1f} min.')
print(f'Final collection size: {coll.count():,}')

## 6. Smoke test on mixed corpus

Same 5 queries as the thesis-only smoke; expect article hits among the top results now.

In [ ]:
queries = [
    'derin öğrenme ile sel tahmini',
    'Türkçe doğal dil işleme metodları',
    'kalp damar hastalıkları teşhis yöntemleri',
    'yenilenebilir enerji rüzgar türbin',
    'üniversite öğrencilerinin akademik başarısı',
]
for q in queries:
    res = coll.query(query_texts=[q], n_results=5)
    metas = res['metadatas'][0]
    dists = res['distances'][0]
    print(f'\nq: {q!r}')
    for m, d in zip(metas, dists):
        src = m.get('source_type') or ('article' if m.get('tez_no','').startswith('art-') else 'thesis')
        title = m.get('title_tr', '')[:80]
        print(f'  [{d:.3f}] [{src:7s}] {m.get("author","?")} ({m.get("year","?")}) — {title}')

## 7. Push the updated Chroma back to HF Hub

Overwrites the existing `chroma_db/` folder in the same dataset repo. Older clients still work; new clients will pull the unified 740K corpus.

In [ ]:
from huggingface_hub import HfApi

api = HfApi()

size_gb = sum(
    os.path.getsize(os.path.join(r, f))
    for r, _, files in os.walk(PERSIST_DIR) for f in files
) / 1024**3
print(f'Uploading updated chroma_db ({size_gb:.1f} GB) ...')

api.upload_folder(
    folder_path=PERSIST_DIR,
    path_in_repo='chroma_db',
    repo_id=HF_REPO,
    repo_type='dataset',
)
print('\n✓ Done. https://huggingface.co/datasets/' + HF_REPO)